# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf

from src.utils.logger import setup_logging


setup_logging()

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.notebook_setup import init_nlp_notebook #noqa E402


cfg = init_nlp_notebook()

if "paths" not in cfg:
    cfg.paths = OmegaConf.create()
cfg.paths.data_dir = str(PROJECT_ROOT / "data")

device = "cuda" if torch.cuda.is_available() else "cpu"

NLP Environment ready. Root: c:\nlp_template_decoder


# Data & Tokenizer

In [2]:
from src.core.data.builder import NLPDataModule


tokenizer = instantiate(cfg.model.tokenizer).build()
tokenizer.padding_side = "left"

datamodule = NLPDataModule(data_cfg=cfg.data, tokenizer=tokenizer)
datamodule.prepare_data()
datamodule.setup(stage="test")

# Вытаскиваем нужные колонки
text_col = cfg.data.get("prompt_column") or cfg.data.get("text_column")
target_col = cfg.data.get("target_column", "target")

test_dataset = datamodule.test_dataset
print(f"Test dataset size: {len(test_dataset)}")

c:\nlp_template_decoder\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:src.core.models.tokenization:Загрузка токенизатора: HuggingFaceM4/tiny-random-LlamaForCausalLM
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got -1. This may result in unexpected behavior.
INFO:src.core.data.builder:Нашли кэш обработанных данных: c:\nlp_template_decoder\data/processed\sft_dataset_processed_b79be0ab. Подготовка пропущена.


Test dataset size: 10


# Load Fine-Tuned Model

In [ ]:
from src.core.models.generator import HFTextGenerator
from src.core.prompts.manager import PromptManager


# Укажи путь к папке с сохраненным адаптером (например, из MLflow артефактов)
LORA_PATH = cfg.get("ckpt_path", "./models/lora_adapter")

print(f"Загрузка модели с адаптером из: {LORA_PATH}")

# Переопределяем путь для загрузки весов
cfg.model.builder.lora_resume_path = LORA_PATH

model_builder = instantiate(cfg.model.builder, tokenizer=tokenizer)
model = model_builder.build()
model.eval()

# Инициализируем генератор и менеджер промптов
generator = HFTextGenerator(
    model=model,
    tokenizer=tokenizer,
    generation_kwargs=cfg.generation_kwargs,
    cleaner_cfg=cfg.model.get("cleaner")
)

prompt_manager = PromptManager(templates=OmegaConf.to_container(cfg.prompts, resolve=True))

Загрузка модели с адаптером из: None


INFO:src.core.models.builder:Загрузка базовой архитектуры: HuggingFaceM4/tiny-random-LlamaForCausalLM
INFO:src.core.models.builder:Применение квантизации BitsAndBytes.
[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 21/21 [00:00<00:00, 1499.96it/s]
INFO:src.core.models.builder:Активация Gradient Checkpointing (Экономия VRAM).
INFO:src.core.models.builder:Режим PEFT: Инициализация нового LoRA адаптера.
INFO:src.core.models.builder:LoRA: 11,776 обучаемых из 1,044,048 (1.1279%)
INFO:src.core.prompts.manager:Инициализирован PromptManager. Загружено шаблонов: 3


# Batch Generation

In [5]:
from hydra.utils import instantiate
from tqdm.auto import tqdm


all_preds = []
all_refs = []
all_prompts = []

# Определяем названия колонок
text_col = cfg.data.get("prompt_column") or cfg.data.get("text_column")
target_col = cfg.data.get("target_column", "completion")

# 1. Загружаем сырой датасет в обход пайплайна токенизации
fetcher = instantiate(cfg.data.source)
raw_dataset = fetcher.load()

# Берем тестовый сплит (с фолбэком на validation/train, если test отсутствует)
if "test" in raw_dataset:
    raw_test_dataset = raw_dataset["test"]
elif "validation" in raw_dataset:
    raw_test_dataset = raw_dataset["validation"]
else:
    raw_test_dataset = raw_dataset["train"]

# 2. Оборачиваем сырые тексты в промпты
for row in raw_test_dataset:
    try:
        all_prompts.append(prompt_manager.render("summarization", text=row[text_col]))
    except ValueError:
        all_prompts.append(row[text_col])
    all_refs.append(row[target_col])

# 3. Батчевая генерация
batch_size = 8
for i in tqdm(range(0, len(all_prompts), batch_size), desc="Evaluating Generation"):
    batch_prompts = all_prompts[i : i + batch_size]

    batch_preds = generator.generate(texts=batch_prompts)
    all_preds.extend(batch_preds)

INFO:src.core.data.fetcher:HF датасет найден локально: c:\nlp_template_decoder\data/raw\HuggingFaceH4_testing_alpaca_small
Evaluating Generation: 100%|██████████| 13/13 [01:22<00:00,  6.32s/it]


# ROUGE Metrics

In [6]:
from torchmetrics.text.rouge import ROUGEScore


rouge_metric = ROUGEScore()
rouge_metric.update(all_preds, all_refs)
results = rouge_metric.compute()

print("\n=== FINAL ROUGE SCORES ===")
for metric_name, tensor_val in results.items():
    print(f"{metric_name}: {tensor_val.item():.4f}")


=== FINAL ROUGE SCORES ===
rouge1_fmeasure: 0.0038
rouge1_precision: 0.0030
rouge1_recall: 0.0066
rouge2_fmeasure: 0.0000
rouge2_precision: 0.0000
rouge2_recall: 0.0000
rougeL_fmeasure: 0.0035
rougeL_precision: 0.0028
rougeL_recall: 0.0061
rougeLsum_fmeasure: 0.0036
rougeLsum_precision: 0.0029
rougeLsum_recall: 0.0063


# Error Analysis (Heuristics)
Анализируем типичные проблемы:
1. **Empty/Truncated Responses:** Ответы обрезаны лимитом токенов или возвращены пустыми.
2. **Repetitions:** Зацикливание модели на одних и тех же фразах.
3. **Worst ROUGE:** Худшие предсказания по метрике для оценки галлюцинаций.

In [ ]:
import pandas as pd


# Считаем ROUGE-L для каждого сэмпла индивидуально для сортировки
sample_metrics = []
for p, r in zip(all_preds, all_refs): #noqa B905
    # Упрощенный расчет индивидуального ROUGE-L (через torchmetrics или evaluate)
    # Для скорости просто запишем метрику длин
    sample_metrics.append({
        "pred_len": len(p.split()),
        "ref_len": len(r.split()),
        "pred": p,
        "ref": r
    })

df_analysis = pd.DataFrame(sample_metrics)

print("--- АНАЛИЗ ПУСТЫХ И КОРОТКИХ ОТВЕТОВ ---")
empty_preds = df_analysis[df_analysis["pred_len"] < 3]
print(f"Пустых/слишком коротких ответов: {len(empty_preds)} из {len(df_analysis)}")

print("\n--- АНАЛИЗ ЗАЦИКЛИВАНИЯ ---")
# Простая эвристика: если длина предсказания в 3+ раза больше референса, возможен цикл
repetitions = df_analysis[df_analysis["pred_len"] > (df_analysis["ref_len"] * 3)]
print(f"Подозрение на зацикливание (очень длинные ответы): {len(repetitions)} из {len(df_analysis)}")
if not repetitions.empty:
    print(f"Пример:\n{repetitions.iloc[0]['pred'][:300]}...")

print("\n--- СЛУЧАЙНЫЕ ПРИМЕРЫ СРЕДНЕГО КАЧЕСТВА ---")
sample_cases = df_analysis.sample(min(3, len(df_analysis)))
for _idx, row in sample_cases.iterrows():
    print(f"\n[ОЖИДАЛОСЬ]:\n{row['ref']}")
    print(f"\n[ПРЕДСКАЗАНО]:\n{row['pred']}")
    print("-" * 50)

--- АНАЛИЗ ПУСТЫХ И КОРОТКИХ ОТВЕТОВ ---
Пустых/слишком коротких ответов: 0 из 100

--- АНАЛИЗ ЗАЦИКЛИВАНИЯ ---
Подозрение на зацикливание (очень длинные ответы): 51 из 100
Пример:
Českcompos header simpl рос LandesornoLines traduditbageчасheadституcred oreCarydro vendмб zugberry двиII dealing EnglandszágCookie cientímar expondireめourgteger Salvador золоtayellow Land starkHowever Ні FileiventIA登 года Schweiz wobei además również Architectureiseconds jun импе bes Auß charged wo...

--- СЛУЧАЙНЫЕ ПРИМЕРЫ СРЕДНЕГО КАЧЕСТВА ---

[ОЖИДАЛОСЬ]:
An example of a data mining problem consisting of open-ended generation is text summarization. Using techniques such as machine learning and natural language processing, a text summarization system can automatically create summaries of long text documents by extracting the important facts and ideas and condensing them into a shorter, more readable version.

[ПРЕДСКАЗАНО]:
varscul просня awardsèg akt дека satisf transparent Ehe Bilditect dort lin swapap